# Mai Chi Flow — complete student strictly below 10M parameters

**Student:** compact Diffusion Transformer (DiT) trained with optimal-transport conditional flow matching, plus a separately trained compact HiFi-GAN-style vocoder. **Teacher data:** ZeroTTS fixed `maichi` voice. **Output:** 24 kHz mono.

This is an experimental budget-constrained implementation inspired by F5-TTS/Matcha-TTS methodology. It is **not** a pretrained Pocket model, a reproduction of those papers, or a proven best-quality model. The previous VITS implementation is retained as a legacy baseline. The updated main path is the modern conditional-flow student in `maichi_flow.py`.

**Measured complete package:** 9,518,564 parameters at vocabulary size 256; 9,551,332 at vocabulary size 512. These counts include the text encoder, duration/prior prediction, DiT, auxiliary pitch head, and **all waveform-generation weights**. No external neural decoder is hidden from the budget. Initialization, export and reload fail at 10,000,000 parameters or more. Quantization does not reduce parameter count.

Run from the top. No base64 source payloads. Full training and quality evaluation must run on your GPU; the included real-audio smoke checks do not establish student quality.

## 1. Configure the GPU run

Start with `pilot`. It validates the pipeline, not speech quality. Use `full` for the main experiment, then continue the same run if validation shows it has not converged. Checkpoint resume restores optimizer/RNG state, but does not replay the exact same data iterator.

For an external RTX PRO 6000 96GB connected through Colab/Jupyter, use a CUDA-enabled PyTorch runtime that supports the actual GPU. Set `USE_DRIVE=False` when Google Drive mounting is unavailable and point `PERSISTENT_BASE` at a persistent disk. Keep at least approximately 60 GB free for the full corpus, feature cache and checkpoints; actual usage varies.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json

MODE = "pilot"                 # "pilot" or "full"
USE_DRIVE = True                # False for an external/local Jupyter runtime
PERSISTENT_BASE = "/content/maichi_flow_v02"  # Set a persistent disk path when USE_DRIVE=False
TEACHER_DEVICE = "cuda"         # Explicitly change to "cpu" if ORT CUDA cannot run
TRAIN_DEVICE = "cuda"
CPU_THREADS = 4
BATCH_SIZE = 32
FEATURE_WORKERS = 4
TRAIN_WORKERS = 4               # Use 0 if multiprocessing is unavailable
RUN_ASR_AUDIT = True
RUN_ABLATION = False             # Also run the same-size student with auxiliary pitch weight zero
RUN_BENCHMARK = MODE == "full"  # Full 137-item evaluation after training
RUN_UTMOS = False                # Optional dependency; skipped values remain missing

assert MODE in {"pilot", "full"}
TEXT_COUNT = 512 if MODE == "pilot" else 20000
VOCODER_STEPS = 2000 if MODE == "pilot" else 100000
FLOW_STEPS = 4000 if MODE == "pilot" else 200000
FLOW_NFE = [4, 8, 16]  # Preset comparison grid; do not tune on the test set.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSISTENT_BASE = "/content/drive/MyDrive/maichi_flow_v02"
ROOT = Path(PERSISTENT_BASE) / MODE
ROOT.mkdir(parents=True, exist_ok=True)
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(WORK)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print({"root": str(ROOT), "mode": MODE, "sentences": TEXT_COUNT, "vocoder_steps": VOCODER_STEPS, "flow_steps": FLOW_STEPS})
print("Free local GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))


## 2. Clone the repository and record the exact source

`REPO_REF="main"` fetches the current code; set a recorded commit SHA for reproduction. A clean checkout is updated safely. The source checksum covers both the shared data pipeline and the new flow implementation. A changed implementation requires a separate run directory or the original commit.

In [ ]:
REPO_URL = "https://github.com/Huyvux12/Ancaptritue.git"
REPO_REF = "main"  # Set a commit SHA for an exact reproduction.
REPO_DIR = WORK / "Ancaptritue"

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), "Checkout belongs to a different repository."
    dirty = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True)
    assert not dirty.strip(), "Checkout has local edits. Preserve them before updating, or choose another REPO_DIR."
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_SHA = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
import hashlib
SOURCE_FILES = ["maichi_pipeline.py", "maichi_flow.py"]
SOURCE_HASHES = {name: hashlib.sha256((REPO_DIR / name).read_bytes()).hexdigest() for name in SOURCE_FILES}
PIPELINE_SHA = hashlib.sha256(json.dumps(SOURCE_HASHES, sort_keys=True).encode()).hexdigest()
revision_path = ROOT / "source_revision.json"
if revision_path.exists():
    previous = json.loads(revision_path.read_text())
    assert previous["pipeline_sha256"] == PIPELINE_SHA, "Pipeline changed since this run. Use the recorded REPO_REF or a new run directory."
revision_path.write_text(json.dumps({"repository": REPO_URL, "requested_ref": REPO_REF,
                                    "commit": REPO_SHA, "pipeline_sha256": PIPELINE_SHA, "files": SOURCE_HASHES}, indent=2))
print("Source commit:", REPO_SHA)


## 3. Dependencies and actual CUDA checks

Keep the runtime's CUDA PyTorch. Install `zerotts` without dependencies so it does not add the CPU ONNX package beside the GPU package. WORLD is compiled against the active NumPy. If a scientific package was already imported in this kernel before installation, restart once and rerun from the top.

The inference package downloads no pretrained acoustic/vocoder weights for the student. Only ZeroTTS and external evaluators are pretrained.

In [ ]:
def pip_install(*packages, extra=None):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(packages)
    if extra:
        cmd += list(extra)
    subprocess.run(cmd, check=True)

pip_install("setuptools==80.9.0", "wheel", "cython==3.0.12", "numpy==2.2.6")
pip_install(extra=["-r", str(REPO_DIR / "requirements-colab.txt")])
os.environ["CC"], os.environ["CXX"] = "gcc", "g++"
pip_install("pyworld==0.3.5", extra=["--no-build-isolation"])
pip_install("zerotts==0.1.5", extra=["--no-deps"])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"], check=True)
if TEACHER_DEVICE == "cuda":
    pip_install("onnxruntime-gpu>=1.23.2,<2")
else:
    pip_install("onnxruntime==1.23.2")


In [ ]:
import torch, numpy as np, onnxruntime as ort
assert torch.cuda.is_available(), "No GPU visible to this kernel. Connect the intended GPU runtime first."
# Real CUDA operations catch missing kernel/architecture support early.
x = torch.randn(512, 512, device="cuda")
_ = x @ x
_ = torch.nn.Conv1d(16, 16, 3).cuda()(torch.randn(2, 16, 128, device="cuda"))
torch.cuda.synchronize()
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Torch/CUDA:", torch.__version__, torch.version.cuda)
print("ONNX providers:", ort.get_available_providers())
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()  # Reuse compatible CUDA/cuDNN libraries loaded by PyTorch.
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
(ROOT / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


## 4. Load source, run CPU regression checks and enforce the budget

The regression checks exercise parameter accounting and rejection, DiT gradient flow, monotonic alignment, Vietnamese tone-bearing characters, padding, BF16 forward, flow integration and exact deployment reload. They are implementation checks, not listening or benchmark results.

All student parameters and exported state tensors must remain below 10M. Discriminators and EMA shadow copies exist only during training and are not part of the inference package.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
for module in ["maichi_pipeline", "maichi_flow"]:
    sys.modules.pop(module, None)
import maichi_pipeline as mp
import maichi_flow as mf
assert Path(mf.__file__).resolve() == (REPO_DIR / "maichi_flow.py").resolve()
for filename in ["maichi_pipeline.py", "maichi_flow.py", "LICENSE_VITS.txt"]:
    shutil.copy2(REPO_DIR / filename, ROOT / filename)

def launch(filename, stage, **kwargs):
    cmd = [sys.executable, str(REPO_DIR / filename), stage, "--root", str(ROOT)]
    for key, value in kwargs.items():
        flag = "--" + key.replace("_", "-")
        if isinstance(value, bool):
            if value:
                cmd.append(flag)
        elif value is not None:
            cmd += [flag, str(value)]
    print("Running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

def data_run(stage, **kwargs):
    return launch("maichi_pipeline.py", stage, **kwargs)

def flow_run(stage, **kwargs):
    return launch("maichi_flow.py", stage, **kwargs)

subprocess.run([sys.executable, str(REPO_DIR / "test_maichi_flow.py")], check=True)
flow_run("budget", vocab_size=256)


## 5. Verify the teacher and reserve all benchmark text

First synthesize three Mai Chi sentences and listen. Then download the 137-item benchmark before selecting any training text. Exact/substring exclusions and deterministic train/val/test splits are applied. This does not rule out paraphrases or shared-source overlap.

In [ ]:
data_run("smoke", device=TEACHER_DEVICE, threads=CPU_THREADS)
from IPython.display import Audio, display
for row in json.loads((ROOT / "smoke.json").read_text())["samples"]:
    print(row["text"])
    display(Audio(filename=row["wav"]))
data_run("benchmark")


## 6. Select transcripts and synthesize Mai Chi audio

Text is sampled through revision-checked Hugging Face JSON pages; ZeroSpeech audio is not downloaded. Every accepted sentence is regenerated with `maichi`. Completed WAVs and sidecars are reused on a rerun. Generation is currently single-request and sequential; a large GPU memory budget alone does not guarantee high teacher throughput.

ZeroSpeech/ZeroBench use CC-BY-NC-4.0, and voice/source rights remain separate from the code license.

In [ ]:
data_run("texts", count=TEXT_COUNT)
data_run("generate", device=TEACHER_DEVICE, threads=CPU_THREADS)
print(json.loads((ROOT / "corpus_summary.json").read_text()))


## 7. Audit data and extract acoustic features

WORLD extracts F0/voicing; the optional pitch objective supervises the text representation, including within-token position for changing pitch contours. It is an auxiliary training signal, not a verified lexical-tone recognizer. A same-parameter-count pitch-weight-zero ablation is optional.

Mel normalization statistics are computed **only from training examples**. Vocabulary is also built only from training text. No old VITS latent features or checkpoints are used by this flow student.

In [ ]:
if RUN_ASR_AUDIT:
    data_run("audit", count=32 if MODE == "full" else 8, device="cuda")
    import pandas as pd
    display(pd.read_json(ROOT / "corpus_asr_audit.jsonl", lines=True).sort_values("wer", ascending=False).head(8))
data_run("features", workers=FEATURE_WORKERS)
vocab = json.loads((ROOT / "vocab.json").read_text())
budget_model = mf.FlowStudent(len(vocab))
actual_budget = mf.assert_budget(budget_model)
print(actual_budget)
mp.atomic_json(ROOT / "student_parameter_budget.json", actual_budget)
del budget_model


## 8. Stage A — train the 1.72M-parameter waveform generator

Train the compact vocoder on real Mai Chi teacher mels and WAVs using spectral, mel, GAN and feature-matching objectives. The discriminator is training-only. This stage must complete before acoustic training; the acoustic run loads its EMA vocoder weights.

No downloaded neural codec or larger frozen vocoder is used at inference. The acoustic and vocoder modules together must fit the strict <10M budget.

In [ ]:
flow_run("train", phase="vocoder", run="vocoder", steps=VOCODER_STEPS,
         batch_size=BATCH_SIZE, workers=TRAIN_WORKERS, device=TRAIN_DEVICE,
         precision=PRECISION, save_every=500 if MODE == "pilot" else 1000,
         eval_every=500 if MODE == "pilot" else 1000, allow_small=MODE == "pilot")


## 9. Stage B — train the conditional-flow acoustic model

A compact text encoder predicts a mel prior and duration. MAS supplies monotonic training alignment; an early diagonal preference decays away over 10,000 steps. A 10-block, width-192 DiT learns the OT-CFM velocity in normalized mel space. At inference, text predicts duration; an ODE sampler generates mel; the trained compact vocoder produces waveform.

This differs from the old VITS posterior/latent-flow/GAN speech model. The waveform decoder is still a convolutional vocoder, a common component in modern mel-based TTS. The implementation is a small experimental adaptation, not an official F5/Matcha/ZipVoice checkpoint. More data/steps or different hyperparameters may be necessary for acceptable quality.

In [ ]:
EXPERIMENTS = [("flow10m", 0.2)]
if RUN_ABLATION:
    EXPERIMENTS.append(("flow10m_no_pitch", 0.0))
for name, pitch_weight in EXPERIMENTS:
    flow_run("train", phase="acoustic", run=name, vocoder_run="vocoder", steps=FLOW_STEPS,
             batch_size=BATCH_SIZE, workers=TRAIN_WORKERS, device=TRAIN_DEVICE,
             precision=PRECISION, pitch_weight=pitch_weight,
             save_every=500 if MODE == "pilot" else 1000,
             eval_every=500 if MODE == "pilot" else 1000, allow_small=MODE == "pilot")


## 10. Inspect loss curves and listen to exported students

`best.pt` is selected by validation loss, not test quality. `student_deploy.pt` contains exactly one acoustic/vocoder state, the vocabulary and normalization buffers. Export and reload both enforce the budget. No teacher duration, F0, audio or latent is used for free-running synthesis.

Pilot speech is not expected to sound natural. The 4/8/16-step samples compare inference effort at the same parameter count, not additional model sizes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
for name in ["vocoder"] + [e[0] for e in EXPERIMENTS]:
    history = pd.read_json(ROOT / "flow_runs" / name / "train.jsonl", lines=True)
    columns = [c for c in ["loss", "flow", "prior", "duration", "pitch", "mel", "stft"] if c in history]
    history.plot(x="step", y=columns, subplots=True, figsize=(10, 2 * len(columns)), title=name)
    plt.tight_layout(); plt.show()
for name, _ in EXPERIMENTS:
    path = ROOT / "flow_runs" / name / "student_deploy.pt"
    model, vocabulary = mf.load_deployment(path, TRAIN_DEVICE)
    print(name, mf.assert_budget(model), "MiB:", round(path.stat().st_size / 2**20, 2))
    text = "Xin chào, hôm nay chúng ta cùng tìm hiểu trí tuệ nhân tạo."
    ids = mp.encode(text, vocabulary)[None].to(TRAIN_DEVICE)
    for nfe in FLOW_NFE:
        audio = model.speak(ids, torch.tensor([ids.shape[1]], device=TRAIN_DEVICE), steps=nfe)[0, 0].float().cpu().numpy()
        print("Flow steps:", nfe)
        display(Audio(audio, rate=mp.SR))
    del model
    torch.cuda.empty_cache()


## 11. Fixed-grid teacher/student benchmark

All 137 items receive raw text; every generated voice is fixed Mai Chi. This is not the original zero-shot voice-cloning task. `cross_lingual` remains an item-origin label, not a tested ability to clone foreign speakers.

`FLOW_NFE=[4,8,16]` is declared before testing. Report the entire trade-off grid; choose hyperparameters using validation data. Measure speed on the same device/thread settings. Teacher is native 48 kHz, student 24 kHz; disclose this difference. Every RTF includes waveform generation. In this implementation one flow step is one velocity-network evaluation, without CFG doubling.

In [ ]:
BENCH_DEVICE = "cuda"  # Use "cpu" for both teacher/student to measure a same-CPU comparison.
BENCH_NAMES = [f"{name}_nfe{nfe}" for name, _ in EXPERIMENTS for nfe in FLOW_NFE]
if RUN_BENCHMARK:
    data_run("synthesize", model="teacher", device=BENCH_DEVICE, threads=CPU_THREADS)
    for name, _ in EXPERIMENTS:
        for nfe in FLOW_NFE:
            flow_run("synthesize", run=name, nfe=nfe, device=BENCH_DEVICE, threads=CPU_THREADS)


## 12. Official quality scorer and adapted Mai Chi similarity

The official scorer retains the two ASR transcripts, expanded references, per-sample WER and silence metrics. UTMOSv2 is optional and missing values remain missing. Its original reference-speaker similarity is invalid for fixed Mai Chi and is not included as Mai Chi similarity in the comparison table. A separate WavLM cosine compares student to teacher Mai Chi on the same text.

No score is invented for steps that have not been executed. Automatic metrics do not replace blind listening, particularly for Vietnamese lexical tone.

In [ ]:
if RUN_UTMOS and RUN_BENCHMARK:
    pip_install("git+https://github.com/sarulab-speech/UTMOSv2.git")
    (ROOT / "pip_freeze_with_utmos.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
if RUN_BENCHMARK:
    for name in ["teacher_maichi"] + BENCH_NAMES:
        data_run("score", name=name, device="cuda", with_utmos=RUN_UTMOS)
    for name in BENCH_NAMES:
        data_run("similarity", name=name, device="cuda")
    data_run("report")
    display(pd.read_csv(ROOT / "comparison.csv"))


## 13. Blinded listening worksheet

Use 30 unique texts per comparison, randomized A/B order, separate keys, and report the listener protocol. Ask about intelligibility, tone errors, naturalness and voice identity. No quality-equivalence claim is supported until actual results are collected.

In [ ]:
if RUN_BENCHMARK:
    import random
    rng = random.Random(2026)
    manifest = mp.read_jsonl(ROOT / "benchmark_manifest.jsonl")
    unique = list({mp.text_key(r["text"]): r for r in manifest}.values())
    chosen = rng.sample(unique, min(30, len(unique)))
    for name in BENCH_NAMES:
        folder = ROOT / "listening" / name
        folder.mkdir(parents=True, exist_ok=True)
        ratings, key = [], []
        for i, row in enumerate(chosen):
            a, b = ("teacher_maichi", name) if rng.random() < 0.5 else (name, "teacher_maichi")
            for label, run_name in [("A", a), ("B", b)]:
                shutil.copy2(ROOT / "benchmark_wavs" / run_name / row["output_wav"], folder / f"{i:03d}_{label}.wav")
            ratings.append({"pair": i, "text": row["text"], "A_naturalness_1_to_5": "", "B_naturalness_1_to_5": "",
                            "A_tone_errors": "", "B_tone_errors": "", "preferred": "", "notes": ""})
            key.append({"pair": i, "id": row["id"], "A": a, "B": b})
        pd.DataFrame(ratings).to_csv(folder / "ratings.csv", index=False)
        pd.DataFrame(key).to_csv(folder / "key_keep_separate.csv", index=False)
print("Results directory:", ROOT)


## Files, resume and scope

- Persistent root: `ROOT`, separate from the legacy v1 directory.
- `flow_runs/vocoder/`: vocoder training/EMA/optimizer checkpoints and logs.
- `flow_runs/<run>/`: acoustic training, validation, strict parameter counts and `student_deploy.pt`.
- `student_parameter_budget.json`: exact full-student budget for the actual vocabulary.
- `scores/`, `benchmark_wavs/`, `comparison.csv`, `listening/`: measured results and listening material.
- Resume requires the same data, vocabulary, implementation hash and training configuration; use a new run name when changing an experiment.

**Completed before publication:** CPU real-Mai-Chi-WAV feature extraction, vocoder and acoustic optimization steps, acoustic resume, waveform generation and exact seeded export/reload; parameter-budget rejection and CPU BF16 objective. **Not completed:** full RTX PRO 6000 training, a 137-item quality benchmark for this new student, GPU throughput measurements, or a listening study. The selected design is a plausible modern <10M candidate, not a proven best architecture.

References: [F5-TTS](https://github.com/SWivid/F5-TTS), [Matcha-TTS](https://github.com/shivammehta25/Matcha-TTS), [ZeroTTS](https://github.com/zeroweight-ai/ZeroTTS).